In [46]:
# Re-write directly to output/ so it's served as a download artifact

import os
import numpy as np
import cv2
from PIL import Image, ImageDraw
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection
import random
import warnings

plt.style.use('dark_background')
print("Рабочая директория:", os.getcwd())

# os.chdir(os.path.dirname(os.path.abspath(__file__)))
OUTPUT_DIR = 'output'
os.makedirs(OUTPUT_DIR, exist_ok=True)

IMAGE_FILES = [
    '1.jpg', '2.jpg', '3.jpg', '4.jpg',
    '5.jpg', '6.jpg', '7.jpg', '8.jpg',
]

Рабочая директория: d:\Libraries\Documents\VScode\CV_labs_Plotnikov_8E21\Lab_2


In [47]:
# ============================================================
# 0. UTILITIES
# ============================================================

def bgr_to_rgb(img):
    return img[..., ::-1].copy()


def bgr_to_gray(img):
    return (
        0.114 * img[:, :, 0].astype(np.float64)
      + 0.587 * img[:, :, 1].astype(np.float64)
      + 0.299 * img[:, :, 2].astype(np.float64)
    ).astype(np.uint8)


def gray_to_rgb(img):
    return np.stack([img, img, img], axis=2)


def save_grid(imgs, title, fname, cmap=None, labels=None):
    n = len(imgs)
    cols = 4
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 4, rows * 3.2))
    fig.suptitle(title, fontsize=14, fontweight='bold', y=1.01)
    axes = axes.ravel()
    for i, img in enumerate(imgs):
        if cmap:
            axes[i].imshow(img, cmap=cmap)
        else:
            disp = img if img.ndim == 3 else gray_to_rgb(img)
            axes[i].imshow(disp)
        lbl = labels[i] if labels else ('Frame ' + str(i + 1))
        axes[i].set_title(lbl, fontsize=9)
        axes[i].axis('off')
    for j in range(n, len(axes)):
        axes[j].axis('off')
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, fname), dpi=110, bbox_inches='tight')
    plt.close()
    print('  Saved: output/' + fname)

In [48]:
# ============================================================
# LOAD IMAGES  (the only cv2 call in the entire script)
# ============================================================

images_color_bgr = []
images_color_rgb = []
images_gray = []

for path in IMAGE_FILES:
    img = cv2.imread(path)
    if img is None:
        raise FileNotFoundError('Cannot load: ' + path)
    images_color_bgr.append(img)
    images_color_rgb.append(bgr_to_rgb(img))
    images_gray.append(bgr_to_gray(img))

print('Loaded ' + str(len(images_gray)) + ' images, shape: ' + str(images_gray[0].shape))

save_grid(
    images_color_rgb, 'Original images (RGB)', '01_original_images.png',
    labels=['Frame ' + str(i + 1) for i in range(8)]
)
save_grid(
    images_gray, 'Grayscale (ITU-R BT.601)', '02_grayscale_images.png',
    cmap='gray', labels=['Frame ' + str(i + 1) for i in range(8)]
)

Loaded 8 images, shape: (720, 1280)
  Saved: output/01_original_images.png
  Saved: output/02_grayscale_images.png


In [49]:
# ============================================================
# STEP 2a. CLAHE - manual implementation
# ============================================================

def clahe_manual(gray, clip_limit=2.0, tile_size=8):
    """
    Contrast Limited Adaptive Histogram Equalization (manual).
    Each tile_size x tile_size block is equalized independently.
    The histogram is clipped at clip_limit*(n_pixels/256);
    the excess is redistributed uniformly across all 256 bins.
    """
    h, w = gray.shape
    out = np.zeros_like(gray, dtype=np.uint8)
    th = (h + tile_size - 1) // tile_size
    tw = (w + tile_size - 1) // tile_size
    for tr in range(th):
        for tc in range(tw):
            y0 = tr * tile_size
            y1 = min((tr + 1) * tile_size, h)
            x0 = tc * tile_size
            x1 = min((tc + 1) * tile_size, w)
            tile = gray[y0:y1, x0:x1].astype(np.int32)
            n_pix = tile.size
            hist = np.bincount(tile.ravel(), minlength=256).astype(np.float64)
            clip = clip_limit * n_pix / 256.0
            excess = np.maximum(0.0, hist - clip).sum()
            hist = np.minimum(hist, clip)
            hist += excess / 256.0
            cdf = np.cumsum(hist)
            cdf_min = cdf[cdf > 0][0]
            denom = max(float(n_pix) - cdf_min, 1.0)
            lut = np.round((cdf - cdf_min) / denom * 255.0).clip(0, 255).astype(np.uint8)
            out[y0:y1, x0:x1] = lut[tile]
    return out


all_images_eq = [clahe_manual(g) for g in images_gray]
print('CLAHE done')

save_grid(
    all_images_eq, 'Brightness equalisation - CLAHE', '03_clahe_equalized.png',
    cmap='gray', labels=['Frame ' + str(i + 1) for i in range(8)]
)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Frame 1: preprocessing stages', fontsize=13, fontweight='bold')
axes[0].imshow(images_color_rgb[0])
axes[0].set_title('Original (RGB)')
axes[1].imshow(images_gray[0], cmap='gray')
axes[1].set_title('Grayscale (BT.601)')
axes[2].imshow(all_images_eq[0], cmap='gray')
axes[2].set_title('After CLAHE')
for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, '04_preprocessing_compare.png'), dpi=110, bbox_inches='tight')
plt.close()
print('  Saved: output/04_preprocessing_compare.png')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('Brightness histograms - Frame 1', fontsize=13, fontweight='bold')
axes[0].hist(images_gray[0].ravel(), bins=256, range=(0, 255), color='#4c78a8', alpha=0.85)
axes[0].set_title('Before CLAHE')
axes[0].set_xlabel('Brightness')
axes[0].set_ylabel('Pixel count')
axes[1].hist(all_images_eq[0].ravel(), bins=256, range=(0, 255), color='#f58518', alpha=0.85)
axes[1].set_title('After CLAHE')
axes[1].set_xlabel('Brightness')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, '05_histograms.png'), dpi=110, bbox_inches='tight')
plt.close()
print('  Saved: output/05_histograms.png')

CLAHE done
  Saved: output/03_clahe_equalized.png
  Saved: output/04_preprocessing_compare.png
  Saved: output/05_histograms.png


In [50]:
# ============================================================
# STEP 1+2b. DoG KEYPOINT DETECTOR + ADAPTIVE NMS
# ============================================================

def gaussian_kernel_1d(sigma):
    radius = max(1, int(3.0 * sigma + 0.5))
    x = np.arange(-radius, radius + 1, dtype=np.float64)
    k = np.exp(-0.5 * (x / sigma) ** 2)
    return k / k.sum()


def gaussian_blur_manual(img, sigma):
    """Separable 1-D Gaussian: horizontal pass then vertical pass."""
    k = gaussian_kernel_1d(sigma)
    r = len(k) // 2
    img_f = img.astype(np.float64)
    padded = np.pad(img_f, [(0, 0), (r, r)], mode='reflect')
    h_out = sum(k[i] * padded[:, i:i + img_f.shape[1]] for i in range(len(k)))
    padded = np.pad(h_out, [(r, r), (0, 0)], mode='reflect')
    return sum(k[i] * padded[i:i + h_out.shape[0], :] for i in range(len(k)))


def build_dog_pyramid(gray, sigmas=(1.0, 1.6, 2.5, 4.0, 6.3, 10.0)):
    """
    Difference-of-Gaussians pyramid.
    DoG approximates the scale-normalised Laplacian - an efficient blob detector.
    """
    blurs = [gaussian_blur_manual(gray, s) for s in sigmas]
    return [blurs[i + 1] - blurs[i] for i in range(len(blurs) - 1)]


def find_local_maxima(dog, threshold=4.0, min_dist=12):
    """
    Scan |DoG| with a sliding window of width 2*min_dist.
    Keep a pixel only if it is the unique maximum in its window
    AND exceeds the threshold.  Step = min_dist//2 overlaps windows.
    """
    h, w = dog.shape
    abs_dog = np.abs(dog)
    kps = []
    step = max(1, min_dist // 2)
    for y in range(min_dist, h - min_dist, step):
        for x in range(min_dist, w - min_dist, step):
            patch = abs_dog[y - min_dist:y + min_dist,
                            x - min_dist:x + min_dist]
            lmax = float(np.max(patch))
            center = float(abs_dog[y, x])
            if center == lmax and center > threshold:
                kps.append((x, y, center))
    return kps


def adaptive_nms(keypoints, target_n=400):
    """
    Adaptive NMS: binary search for the minimum suppression radius r*
    that keeps exactly target_n keypoints.
    Points sorted by descending response so stronger ones suppress weaker neighbours.
    """
    if len(keypoints) <= target_n:
        return keypoints
    kps = sorted(keypoints, key=lambda k: -k[2])
    lo = 1.0
    hi = 2000.0
    best = kps[:target_n]
    for _ in range(25):
        mid = (lo + hi) / 2.0
        selected = []
        suppressed = set()
        for i in range(len(kps)):
            if i in suppressed:
                continue
            selected.append(kps[i])
            for j in range(i + 1, len(kps)):
                if j not in suppressed:
                    dx = kps[j][0] - kps[i][0]
                    dy = kps[j][1] - kps[i][1]
                    if dx * dx + dy * dy < mid * mid:
                        suppressed.add(j)
        if len(selected) >= target_n:
            best = selected[:target_n]
            lo = mid
        else:
            hi = mid
    return best


def downsample_2x(gray):
    return gaussian_blur_manual(gray, 1.0)[::2, ::2]


all_keypoints = []
for i in range(8):
    eq = all_images_eq[i]
    raw = []
    for dog in build_dog_pyramid(eq):
        raw.extend(find_local_maxima(dog, threshold=4.0, min_dist=12))
    small = downsample_2x(eq)
    for dog in build_dog_pyramid(small):
        for x, y, r in find_local_maxima(dog, threshold=4.0, min_dist=6):
            raw.append((x * 2, y * 2, r))
    filtered = adaptive_nms(raw, target_n=500)
    all_keypoints.append(filtered)
    print('  Frame ' + str(i + 1) + ': ' + str(len(raw)) + ' raw -> ' + str(len(filtered)) + ' after ANMS')


def draw_keypoints_pil(rgb_img, keypoints):
    """Draw keypoints with PIL ellipses (no cv2.circle)."""
    pil = Image.fromarray(rgb_img)
    draw = ImageDraw.Draw(pil)
    for kp in keypoints:
        x = int(kp[0])
        y = int(kp[1])
        draw.ellipse([x - 6, y - 6, x + 6, y + 6], outline=(0, 255, 0), width=1)
        draw.ellipse([x - 2, y - 2, x + 2, y + 2], fill=(0, 200, 255))
    return np.array(pil)


fig, axes = plt.subplots(2, 4, figsize=(18, 9))
fig.suptitle('Keypoints (DoG + Adaptive NMS)', fontsize=14, fontweight='bold')
axes = axes.ravel()
for i in range(8):
    kp_img = draw_keypoints_pil(images_color_rgb[i], all_keypoints[i])
    axes[i].imshow(kp_img)
    axes[i].set_title('Frame ' + str(i + 1) + ' - ' + str(len(all_keypoints[i])) + ' pts', fontsize=9)
    axes[i].axis('off')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, '06_keypoints_all.png'), dpi=110, bbox_inches='tight')
plt.close()
print('  Saved: output/06_keypoints_all.png')

  Frame 1: 112 raw -> 112 after ANMS
  Frame 2: 125 raw -> 125 after ANMS
  Frame 3: 135 raw -> 135 after ANMS
  Frame 4: 143 raw -> 143 after ANMS
  Frame 5: 101 raw -> 101 after ANMS
  Frame 6: 111 raw -> 111 after ANMS
  Frame 7: 102 raw -> 102 after ANMS
  Frame 8: 106 raw -> 106 after ANMS
  Saved: output/06_keypoints_all.png


In [51]:
# ============================================================
# STEP 3. SIFT DESCRIPTOR (fully manual)
# ============================================================

def compute_gradient(img):
    """Central-difference gradient (2nd-order accurate)."""
    f = img.astype(np.float64)
    gx = np.zeros_like(f)
    gy = np.zeros_like(f)
    gx[:, 1:-1] = (f[:, 2:] - f[:, :-2]) / 2.0
    gx[:, 0] = f[:, 1] - f[:, 0]
    gx[:, -1] = f[:, -1] - f[:, -2]
    gy[1:-1, :] = (f[2:, :] - f[:-2, :]) / 2.0
    gy[0, :] = f[1, :] - f[0, :]
    gy[-1, :] = f[-1, :] - f[-2, :]
    return np.sqrt(gx ** 2 + gy ** 2), np.arctan2(gy, gx)


def dominant_orientation(mag_patch, ori_patch, n_bins=36):
    """
    Weighted gradient-orientation histogram of a patch.
    Weight = gradient magnitude.  Returns angle of the tallest bin (radians).
    """
    hist = np.zeros(n_bins)
    for angle, w in zip(ori_patch.ravel(), mag_patch.ravel()):
        b = int((angle + np.pi) / (2.0 * np.pi) * n_bins) % n_bins
        hist[b] += w
    return (float(np.argmax(hist)) / float(n_bins)) * 2.0 * np.pi - np.pi


def sift_descriptor_manual(eq, x, y, patch_size=16, n_cells=4, n_ori_bins=8):
    """
    Manual 128-D SIFT descriptor.

    1. Extract patch_size x patch_size patch (reflect-padded).
    2. Compute gradient magnitude/orientation.
    3. Find dominant orientation; subtract it (rotation invariance).
    4. Divide into n_cells x n_cells sub-cells; build n_ori_bins histogram each.
    5. Concatenate -> 128-D vector.
    6. L2-normalise -> clip at 0.2 -> L2-normalise again.
    """
    half = patch_size // 2
    pad = half + 2
    padded = np.pad(eq.astype(np.float64), pad, mode='reflect')
    cx = x + pad
    cy = y + pad
    patch = padded[cy - half:cy + half, cx - half:cx + half]
    if patch.shape != (patch_size, patch_size):
        return None
    mag, ori = compute_gradient(patch)
    dom_ori = dominant_orientation(mag, ori)
    ori_rel = ori - dom_ori
    cell = patch_size // n_cells
    desc = []
    for r in range(n_cells):
        for c in range(n_cells):
            m_cell = mag[r * cell:(r + 1) * cell, c * cell:(c + 1) * cell]
            o_cell = ori_rel[r * cell:(r + 1) * cell, c * cell:(c + 1) * cell]
            hist = np.zeros(n_ori_bins)
            for angle, weight in zip(o_cell.ravel(), m_cell.ravel()):
                b = int((angle + np.pi) / (2.0 * np.pi) * n_ori_bins) % n_ori_bins
                hist[b] += weight
            desc.extend(hist.tolist())
    desc = np.array(desc, dtype=np.float32)
    n = float(np.linalg.norm(desc))
    if n > 1e-6:
        desc = desc / n
    desc = np.clip(desc, 0.0, 0.2)
    n2 = float(np.linalg.norm(desc))
    if n2 > 1e-6:
        desc = desc / n2
    return desc


all_descs = []
all_valid_kps = []

for i in range(8):
    eq = all_images_eq[i]
    kps = all_keypoints[i]
    descs = []
    valid = []
    for kp in kps:
        d = sift_descriptor_manual(eq, kp[0], kp[1])
        if d is not None:
            descs.append(d)
            valid.append(kp)
    if descs:
        all_descs.append(np.array(descs, dtype=np.float32))
    else:
        all_descs.append(np.empty((0, 128), dtype=np.float32))
    all_valid_kps.append(valid)
    print('  Frame ' + str(i + 1) + ': ' + str(len(valid)) + ' descriptors')


# Visualise one descriptor
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('SIFT descriptor (128-D) - Frame 1, keypoint 0', fontsize=13, fontweight='bold')
d0_mat = all_descs[0][0].reshape(4, 32)
im = axes[0].imshow(d0_mat, cmap='hot', aspect='auto')
axes[0].set_title('Heat map (4 cell rows x 32 orientation bins)')
axes[0].set_xlabel('Orientation bins')
axes[0].set_ylabel('Cell rows')
plt.colorbar(im, ax=axes[0])
axes[1].bar(np.arange(128), all_descs[0][0], color='#4c78a8', width=0.9)
axes[1].set_title('128-D descriptor vector')
axes[1].set_xlabel('Dimension')
axes[1].set_ylabel('Value')
axes[1].axhline(0.2, color='red', linestyle='--', alpha=0.6, label='Clip threshold 0.2')
axes[1].legend()
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, '07_descriptor_example.png'), dpi=110, bbox_inches='tight')
plt.close()
print('  Saved: output/07_descriptor_example.png')


  Frame 1: 112 descriptors
  Frame 2: 125 descriptors
  Frame 3: 135 descriptors
  Frame 4: 143 descriptors
  Frame 5: 101 descriptors
  Frame 6: 111 descriptors
  Frame 7: 102 descriptors
  Frame 8: 106 descriptors
  Saved: output/07_descriptor_example.png


In [52]:
# ============================================================
# STEP 4. DESCRIPTOR MATCHING (Lowe ratio test, manual BF)
# ============================================================

def match_descriptors_manual(desc1, desc2, ratio_thresh=0.80):
    """
    Brute-force nearest-neighbour matching + Lowe ratio test (2004).
    Match (i, j1) accepted iff dist(i,j1) < ratio_thresh * dist(i,j2).
    """
    if len(desc1) == 0 or len(desc2) == 0:
        return []
    diff = desc1[:, np.newaxis, :] - desc2[np.newaxis, :, :]
    dist_mat = np.sqrt((diff ** 2).sum(axis=2))
    matches = []
    for i in range(len(desc1)):
        row = dist_mat[i]
        idx = np.argsort(row)
        if len(idx) >= 2 and row[idx[0]] < ratio_thresh * row[idx[1]]:
            matches.append((i, int(idx[0]), float(row[idx[0]])))
    return matches


def draw_matches_manual(img1, img2, kps1, kps2, matches, max_draw=40):
    """Visualise correspondences with PIL (replaces cv2.drawMatches)."""
    h1, w1 = img1.shape[:2]
    h2, w2 = img2.shape[:2]
    canvas = Image.new('RGB', (w1 + w2, max(h1, h2)))
    canvas.paste(Image.fromarray(img1), (0, 0))
    canvas.paste(Image.fromarray(img2), (w1, 0))
    draw = ImageDraw.Draw(canvas)
    palette = [
        (255, 80, 80), (80, 255, 80), (80, 130, 255), (255, 210, 0),
        (0, 210, 255), (210, 0, 255), (255, 140, 0), (0, 255, 160),
    ]
    for idx, match in enumerate(matches[:max_draw]):
        i1 = match[0]
        i2 = match[1]
        x1 = int(kps1[i1][0])
        y1 = int(kps1[i1][1])
        x2 = int(kps2[i2][0]) + w1
        y2 = int(kps2[i2][1])
        c = palette[idx % len(palette)]
        draw.ellipse([x1 - 5, y1 - 5, x1 + 5, y1 + 5], outline=c, width=2)
        draw.ellipse([x2 - 5, y2 - 5, x2 + 5, y2 + 5], outline=c, width=2)
        draw.line([(x1, y1), (x2, y2)], fill=c, width=1)
    return canvas


all_matches = []
for i in range(len(all_valid_kps) - 1):
    m = match_descriptors_manual(all_descs[i], all_descs[i + 1])
    all_matches.append(m)
    vis = draw_matches_manual(
        images_color_rgb[i],
        images_color_rgb[i + 1],
        all_valid_kps[i],
        all_valid_kps[i + 1],
        m
    )
    vis.save(os.path.join(OUTPUT_DIR, 'matches_' + str(i + 1) + '_' + str(i + 2) + '.png'))
    print('  Pair ' + str(i + 1) + '->' + str(i + 2) + ': ' + str(len(m)) + ' matches')


fig, axes2 = plt.subplots(4, 2, figsize=(18, 20))
fig.suptitle('Keypoint matching (Lowe ratio test, ratio=0.80)', fontsize=14, fontweight='bold')
axes2 = axes2.ravel()
for i in range(7):
    fpath = os.path.join(OUTPUT_DIR, 'matches_' + str(i + 1) + '_' + str(i + 2) + '.png')
    img_vis = plt.imread(fpath)
    axes2[i].imshow(img_vis)
    axes2[i].set_title(
        'Pair ' + str(i + 1) + '->' + str(i + 2) + ' - ' + str(len(all_matches[i])) + ' matches',
        fontsize=10
    )
    axes2[i].axis('off')
axes2[7].axis('off')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, '08_matches_grid.png'), dpi=80, bbox_inches='tight')
plt.close()
print('  Saved: output/08_matches_grid.png')


  Pair 1->2: 6 matches
  Pair 2->3: 11 matches
  Pair 3->4: 12 matches
  Pair 4->5: 7 matches
  Pair 5->6: 13 matches
  Pair 6->7: 11 matches
  Pair 7->8: 9 matches
  Saved: output/08_matches_grid.png


In [53]:
# ============================================================
# STEP 5. RANSAC - rigid transform (rotation + translation only)
# ============================================================

def estimate_rigid_ransac(kps1, kps2, matches, n_iter=2000, threshold=3.0):
    """
    RANSAC estimation of a 2x3 matrix [R | t] (rotation + translation).

    Minimal sample: 2 point pairs -> analytic solution:
      cos(a) = (dp1 . dp2) / |dp1|^2
      sin(a) = (dp1 x dp2) / |dp1|^2
      t = p2_0 - R * p1_0

    After RANSAC: least-squares refinement on inliers via SVD (Procrustes method).
    """
    if len(matches) < 2:
        return None, 0

    pts1 = np.array(
        [[kps1[m[0]][0], kps1[m[0]][1]] for m in matches], dtype=np.float64
    )
    pts2 = np.array(
        [[kps2[m[1]][0], kps2[m[1]][1]] for m in matches], dtype=np.float64
    )

    best_mask = None
    best_count = 0
    rng = np.random.default_rng(42)

    for _ in range(n_iter):
        idx = rng.choice(len(matches), 2, replace=False)
        dp1 = pts1[idx[0]] - pts1[idx[1]]
        dp2 = pts2[idx[0]] - pts2[idx[1]]
        n1 = float(dp1[0] * dp1[0] + dp1[1] * dp1[1])
        if n1 < 1e-6:
            continue
        cos_a = float(dp1[0] * dp2[0] + dp1[1] * dp2[1]) / n1
        sin_a = float(dp1[0] * dp2[1] - dp1[1] * dp2[0]) / n1
        R = np.array([[cos_a, -sin_a], [sin_a, cos_a]])
        t = pts2[idx[0]] - R @ pts1[idx[0]]
        proj = (R @ pts1.T).T + t
        err = np.sqrt(((proj - pts2) ** 2).sum(axis=1))
        mask = err < threshold
        count = int(mask.sum())
        if count > best_count:
            best_count = count
            best_mask = mask.copy()

    if best_mask is None or int(best_mask.sum()) < 2:
        return None, 0

    # Least-squares refinement on inliers (Procrustes via SVD)
    p1_in = pts1[best_mask]
    p2_in = pts2[best_mask]
    c1 = p1_in.mean(axis=0)
    c2 = p2_in.mean(axis=0)
    H = (p1_in - c1).T @ (p2_in - c2)
    U, _S, Vt = np.linalg.svd(H)
    R_ref = Vt.T @ U.T
    if np.linalg.det(R_ref) < 0:
        Vt[-1, :] = Vt[-1, :] * (-1.0)
        R_ref = Vt.T @ U.T
    t_ref = c2 - R_ref @ c1
    M = np.hstack([R_ref, t_ref.reshape(2, 1)])
    return M, int(best_mask.sum())


transforms = []
tx_list = []
ty_list = []
ang_list = []

for i in range(len(all_matches)):
    M, n_in = estimate_rigid_ransac(
        all_valid_kps[i], all_valid_kps[i + 1], all_matches[i]
    )
    transforms.append(M)
    if M is not None:
        tx = float(M[0, 2])
        ty = float(M[1, 2])
        angle = float(np.degrees(np.arctan2(M[1, 0], M[0, 0])))
        tx_list.append(-tx)
        ty_list.append(-ty)
        ang_list.append(angle)
        print(
            '  ' + str(i + 1) + '->' + str(i + 2) +
            ': tx=' + str(round(tx, 1)) +
            ' ty=' + str(round(ty, 1)) +
            ' angle=' + str(round(angle, 2)) + 'deg' +
            ' inliers=' + str(n_in)
        )
    else:
        tx_list.append(0.0)
        ty_list.append(0.0)
        ang_list.append(0.0)
        print('  ' + str(i + 1) + '->' + str(i + 2) + ': no matches')


  1->2: tx=-328.3 ty=-82.3 angle=-2.03deg inliers=3
  2->3: tx=-5.9 ty=109.3 angle=-22.49deg inliers=4
  3->4: tx=-218.2 ty=329.2 angle=-33.35deg inliers=3
  4->5: tx=149.2 ty=-26.5 angle=-7.94deg inliers=2
  5->6: tx=366.0 ty=-94.0 angle=0.0deg inliers=3
  6->7: tx=32.8 ty=-34.2 angle=12.5deg inliers=6
  7->8: tx=211.9 ty=-287.9 angle=43.2deg inliers=2


In [54]:
# ============================================================
# STEP 6. CAMERA TRAJECTORY
# ============================================================
#
# Image transform  T_img : p' = R*p + t
# Camera motion is its inverse: T_cam = inv(T_img)
# Accumulate poses: pose_{k+1} = pose_k * T_cam_{k->k+1}
# ============================================================

trajectory = [(0.0, 0.0, 0.0)]
pose = np.eye(3)
SCALE = 0.01

for M in transforms:
    if M is not None:
        M_h = np.vstack([M, np.array([0.0, 0.0, 1.0])])
        if abs(np.linalg.det(M_h)) > 1e-6:
            pose = pose @ np.linalg.inv(M_h)
    x_pos = pose[0, 2] * SCALE
    y_pos = pose[1, 2] * SCALE
    a_pos = float(np.arctan2(pose[1, 0], pose[0, 0]))
    trajectory.append((x_pos, y_pos, a_pos))

xs = [t[0] for t in trajectory]
ys = [t[1] for t in trajectory]
angles_traj = [t[2] for t in trajectory]

print('\nCamera trajectory:')
for i in range(len(trajectory)):
    x = trajectory[i][0]
    y = trajectory[i][1]
    a = trajectory[i][2]
    print(
        '  Frame ' + str(i + 1) +
        ': x=' + str(round(x, 2)) +
        ' y=' + str(round(y, 2)) +
        ' angle=' + str(round(float(np.degrees(a)), 1)) + 'deg'
    )


# Final visualisation: trajectory + displacement bars + rotation bars
pair_labels = [str(i + 1) + '->' + str(i + 2) for i in range(7)]
xp = np.arange(7)
bar_w = 0.35

fig = plt.figure(figsize=(18, 14))
fig.suptitle('Visual Odometry - Lab 2', fontsize=15, fontweight='bold')

ax1 = fig.add_subplot(2, 2, (1, 3))
ax1.set_title('Camera trajectory (top-down view)', fontsize=12)
points = np.array([xs, ys]).T.reshape(-1, 1, 2)
segments = np.concatenate([points[:-1], points[1:]], axis=1)
lc = LineCollection(segments, cmap='plasma', linewidth=3)
lc.set_array(np.linspace(0, 1, len(segments)))
ax1.add_collection(lc)
plt.colorbar(lc, ax=ax1, label='Time (start -> end)', shrink=0.7)

for i in range(len(trajectory)):
    xi = trajectory[i][0]
    yi = trajectory[i][1]
    ai = trajectory[i][2]
    if i == 0:
        c = 'limegreen'
        s = 150
    elif i == len(trajectory) - 1:
        c = 'crimson'
        s = 150
    else:
        c = 'steelblue'
        s = 60
    ax1.scatter(xi, yi, c=c, s=s, zorder=6)
    ax1.annotate('  ' + str(i + 1), (xi, yi), fontsize=10, fontweight='bold')
    dx = float(np.cos(ai)) * 0.15
    dy = float(np.sin(ai)) * 0.15
    ax1.annotate(
        '', xy=(xi + dx, yi + dy), xytext=(xi, yi),
        arrowprops=dict(arrowstyle='->', color='#555555', lw=1.5)
    )

ax1.set_aspect('equal')
ax1.grid(True, alpha=0.3)
ax1.margins(0.25)
ax1.scatter([], [], c='limegreen', s=80, label='Start')
ax1.scatter([], [], c='crimson', s=80, label='End')
ax1.legend(fontsize=10)
ax1.set_xlabel('X (units)')
ax1.set_ylabel('Y (units)')

ax2 = fig.add_subplot(2, 2, 2)
ax2.bar(xp - bar_w / 2, tx_list, bar_w, label='DeltaX', color='#4c78a8')
ax2.bar(xp + bar_w / 2, ty_list, bar_w, label='DeltaY', color='#f58518')
ax2.set_xticks(xp)
ax2.set_xticklabels(pair_labels)
ax2.set_title('Camera displacement vector (pixels)', fontsize=11)
ax2.set_xlabel('Frame pair')
ax2.set_ylabel('Pixels')
ax2.axhline(0, color='black', lw=0.8)
ax2.legend()
ax2.grid(True, axis='y', alpha=0.3)

ax3 = fig.add_subplot(2, 2, 4)
ax3.bar(xp, ang_list, color='#72b7b2')
ax3.set_xticks(xp)
ax3.set_xticklabels(pair_labels)
ax3.set_title('Camera rotation angle (degrees)', fontsize=11)
ax3.set_xlabel('Frame pair')
ax3.set_ylabel('Degrees')
ax3.axhline(0, color='black', lw=0.8)
ax3.grid(True, axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, '09_camera_trajectory.png'), dpi=120, bbox_inches='tight')
plt.close()
print('  Saved: output/09_camera_trajectory.png')

# Match count chart
match_counts = [len(m) for m in all_matches]
fig, ax = plt.subplots(figsize=(10, 5))
bar_colors = ['#4c78a8' if n >= 4 else '#f58518' for n in match_counts]
bars = ax.bar(pair_labels, match_counts, color=bar_colors)
for bar, n in zip(bars, match_counts):
    ax.text(
        bar.get_x() + bar.get_width() / 2.0,
        bar.get_height() + 0.1,
        str(n), ha='center', va='bottom', fontweight='bold'
    )
ax.set_title('Keypoint match count per frame pair', fontsize=12)
ax.set_xlabel('Frame pair')
ax.set_ylabel('Number of matches')
ax.axhline(4, color='red', linestyle='--', alpha=0.6, label='Min for RANSAC (4)')
ax.legend()
ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, '10_match_counts.png'), dpi=110, bbox_inches='tight')
plt.close()
print('  Saved: output/10_match_counts.png')

print('\nDone. All files in: ' + os.path.abspath(OUTPUT_DIR))
for f in sorted(os.listdir(OUTPUT_DIR)):
    print('  ' + f)



Camera trajectory:
  Frame 1: x=0.0 y=0.0 angle=0.0deg
  Frame 2: x=3.25 y=0.94 angle=2.0deg
  Frame 3: x=3.76 y=-0.03 angle=24.5deg
  Frame 4: x=7.71 y=0.07 angle=57.9deg
  Frame 5: x=6.85 y=-1.19 angle=65.8deg
  Frame 6: x=4.5 y=-4.14 angle=65.8deg
  Frame 7: x=4.03 y=-4.2 angle=53.3deg
  Frame 8: x=1.43 y=-1.74 angle=10.1deg
  Saved: output/09_camera_trajectory.png
  Saved: output/10_match_counts.png

Done. All files in: d:\Libraries\Documents\VScode\CV_labs_Plotnikov_8E21\Lab_2\output
  01_original_images.png
  02_grayscale_images.png
  03_clahe_equalized.png
  04_preprocessing_compare.png
  05_histograms.png
  06_keypoints_all.png
  07_descriptor_example.png
  08_matches_grid.png
  09_camera_trajectory.png
  10_match_counts.png
  matches_1_2.png
  matches_2_3.png
  matches_3_4.png
  matches_4_5.png
  matches_5_6.png
  matches_6_7.png
  matches_7_8.png
